In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.linear_model import LinearRegression
from scipy.optimize import minimize
import scipy.stats as scst
from scipy.stats import mannwhitneyu
from scipy.stats import ttest_ind
from sklearn.mixture import GaussianMixture
import random
import scipy.stats as stats
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from matplotlib import cm
from matplotlib.colors import Normalize
import statsmodels.api as sm
from scipy.optimize import nnls

In [ ]:
pd.set_option('display.max_columns', None)
plt.rcParams['svg.fonttype'] = 'none'

#### Load dataframe with centromere predictions and CN based classification of chromosomes
- merge into a single dataframe

In [ ]:
# load
pred_df = pd.read_csv("ppcg_prediction.tsv", sep="\t")
class_df = pd.read_csv("/g/korbel/hain/centromere/PPCG_classification.updated.tsv", sep="\t")

# make key
pred_df["KEY"] = pred_df["SAMPLE_TUMOR"] + "_" + pred_df["CHROM"]
class_df["KEY"] = class_df["SAMPLE"] + "_" + class_df["CHROM"]

# merge
pred_df_annot = pd.merge(pred_df, class_df[["CN_CLASS", "LOH_CLASS", "KEY"]], on="KEY", how="inner").copy()

#### Set class for each chromosome for later plotting/stratification...

In [ ]:
col_merged_class = []
col_arm = []
col_project = []

for idx, row in tqdm(pred_df_annot.iterrows(), total=pred_df_annot.shape[0]):

    if row["CN_CLASS"] == "ISO_P":
        col_merged_class.append("ISO")
        col_arm.append("P")
    elif row["CN_CLASS"] == "ISO_Q":
        col_merged_class.append("ISO")
        col_arm.append("Q")
    elif row["CN_CLASS"] == "P_ARM_LOSS":
        col_merged_class.append("ARM_LOSS")
        col_arm.append("P")
    elif row["CN_CLASS"] == "Q_ARM_LOSS":
        col_merged_class.append("ARM_LOSS")
        col_arm.append("Q")
    elif row["CN_CLASS"] == "P_ARM_GAIN":
        col_merged_class.append("ARM_GAIN")
        col_arm.append("P")
    elif row["CN_CLASS"] == "Q_ARM_GAIN":
        col_merged_class.append("ARM_GAIN")
        col_arm.append("Q")
    elif row["CN_CLASS"] == "WHOLE_LOSS":
        col_merged_class.append("WHOLE_LOSS")
        col_arm.append("BOTH")
    elif row["CN_CLASS"] == "WHOLE_GAIN":
        col_merged_class.append("WHOLE_GAIN")
        col_arm.append("BOTH")
    elif row["CN_CLASS"] == "NORMAL":
        col_merged_class.append("NORMAL")
        col_arm.append("BOTH")
    elif row["CN_CLASS"] == "GAIN_4_4":
        col_merged_class.append("GAIN_4")
        col_arm.append("BOTH")
    elif row["CN_CLASS"] == "GAIN_5_5":
        col_merged_class.append("GAIN_5")
        col_arm.append("BOTH")
    elif row["CN_CLASS"] == "GAIN_1_4":
        col_merged_class.append("GAIN_4|1")
        col_arm.append("P")
    elif row["CN_CLASS"] == "GAIN_4_1":
        col_merged_class.append("GAIN_4|1")
        col_arm.append("Q")        
    elif row["CN_CLASS"] == "GAIN_2_4":
        col_merged_class.append("GAIN_4|2")
        col_arm.append("P")
    elif row["CN_CLASS"] == "GAIN_4_2":
        col_merged_class.append("GAIN_4|2")
        col_arm.append("Q")
    elif row["CN_CLASS"] == "GAIN_3_4":
        col_merged_class.append("GAIN_4|3")
        col_arm.append("P")
    elif row["CN_CLASS"] == "GAIN_4_3":
        col_merged_class.append("GAIN_4|3")
        col_arm.append("Q")
    elif row["CN_CLASS"] == "GAIN_5_4":
        col_merged_class.append("GAIN_4|5")
        col_arm.append("P")
    elif row["CN_CLASS"] == "GAIN_4_5":
        col_merged_class.append("GAIN_4|5")
        col_arm.append("Q")
    else:
        col_merged_class.append(row["CN_CLASS"])
        col_arm.append("BOTH")

    if row["HISTOLOGY"] == "PPCG":
        col_project.append("PPCG")
    else:
        col_project.append("PCAWG")

pred_df_annot["MERGED_CN_CLASS"] = col_merged_class
pred_df_annot["ARM"] = col_arm
pred_df_annot["PROJECT"] = col_project

In [ ]:
# plotting orders for full set and set reuduced to major classes
merged_cn_class_order = ["WHOLE_LOSS", "NORMAL", "WHOLE_GAIN", "GAIN_4", "GAIN_5", "ARM_LOSS", "ARM_GAIN", "ISO", "GAIN_4|1", "GAIN_4|2", "GAIN_4|3", "GAIN_4|5", "OTHER"]
merged_cn_class_order_red = ["WHOLE_LOSS", "NORMAL", "WHOLE_GAIN", "GAIN_4", "GAIN_5", "ARM_LOSS", "ARM_GAIN", "ISO"]

<div style="padding: 12px; border-left: 5px solid #f0ad4e; background-color: #fff3cd; color: #664d03;">
  <strong>Warning:</strong> FILTER
</div>

In [ ]:
cutoff_groups = {
    0.12:"GROUP1", 0.07:"GROUP2", 0.2:"GROUP3", 0.3:"GROUP4", 0.4:"GROUP5", 
    0.5:"GROUP6", 0.15:"GROUP7", 0.25:"GROUP8", 0.21:"GROUP9", 0.22:"GROUP10",
    0.23:"GROUP11", 0.24:"GROUP12"
    }

In [ ]:
filter_id = 0.3
pred_df_annot = pred_df_annot[pred_df_annot[cutoff_groups[filter_id]]].copy().reset_index(drop=True)

In [ ]:
pred_df_annot = pred_df_annot.drop_duplicates(subset=["SAMPLE_GERMLINE", "CHROM"], keep="first").reset_index(drop=True)

In [ ]:
pred_df_annot["SAMPLE_TUMOR"].nunique()

In [ ]:
pred_df_annot["SAMPLE_GERMLINE"].nunique()

### Rescale

1.) Fit linear regression to neutral ladder data

In [ ]:
### gather median of each neutral ladder step
x = []
y = []
for y_val, cn_class in zip([0, 1, 2], ["WHOLE_LOSS", "NORMAL", "WHOLE_GAIN"]):
    x.append(pred_df_annot[pred_df_annot["MERGED_CN_CLASS"] == cn_class]["TOTAL_HOR_LENGTH_F_COV_2"].median())
    y.append(y_val)

### fit linear regression
model_ladder = LinearRegression()
model_ladder.fit(np.array(x).reshape(-1, 1), y, sample_weight=[0.2, 1, 0.2])

2.) Rescale F value, add 1 to directly create number of centromeres in cell value

In [ ]:
pred_df_annot["SCALED_F"] = 1 + model_ladder.predict(
    pred_df_annot[["TOTAL_HOR_LENGTH_F_COV_2"]].values
)

# Plot landscape - 4B

Make a dataframe that connects which CN states would fit to each type of event

In [ ]:
cn_df_data = {"CLASS":[], "CN":[]}
for m_cn in pred_df_annot["MERGED_CN_CLASS"].unique():
    cn_df_data["CLASS"].append(m_cn)
    if m_cn == "WHOLE_LOSS":
        cn_df_data["CN"].append(["1_1"])
    elif m_cn == "NORMAL":
        cn_df_data["CN"].append(["2_2"])
    elif m_cn == "WHOLE_GAIN":
        cn_df_data["CN"].append(["3_3"])
    elif m_cn == "GAIN_4":
        cn_df_data["CN"].append(["4_4"])
    elif m_cn == "ARM_LOSS":
        cn_df_data["CN"].append(["2_1", "1_2"])
    elif m_cn == "ARM_GAIN":
        cn_df_data["CN"].append(["2_3", "3_2"])
    elif m_cn == "ISO":
        cn_df_data["CN"].append(["1_3", "3_1"])
    else:
        cn_df_data["CN"].append(["2_2"])        

cn_df = pd.DataFrame(cn_df_data)
cn_df.set_index("CLASS", inplace=True, drop=True)

Check for which chromosomes the predicted event and the predicted CN fit 
- if this does not fit it means that the breakpoint is close to but not in the centromere

In [ ]:
col_correct_CN = []
for _, row in pred_df_annot.iterrows():
    if row["CN"] in cn_df.loc[row["MERGED_CN_CLASS"]]["CN"]:
        col_correct_CN.append("YES")
    else:
        col_correct_CN.append("NO")
pred_df_annot["CORRECT_CN"] = col_correct_CN

#### PLOT - Landscape

In [ ]:
def plot_landscape (axis):
    
    ### hide outliers
    quantile_df_data = {"CLASS":[], "LOWER":[], "UPPER":[]}
    for m_cn in pred_df_annot["MERGED_CN_CLASS"].unique():
        quantile_df_data["CLASS"].append(m_cn)
        quantile_df_data["LOWER"].append(pred_df_annot[pred_df_annot["MERGED_CN_CLASS"] == m_cn]["SCALED_F"].quantile(0.02)) # 0.01
        quantile_df_data["UPPER"].append(pred_df_annot[pred_df_annot["MERGED_CN_CLASS"] == m_cn]["SCALED_F"].quantile(0.98)) # 0.99
    quantile_df = pd.DataFrame(quantile_df_data)
    quantile_df.set_index("CLASS", inplace=True, drop=True)

    ### decide if a datapoint is used for the plot -> depends on matching CN and classification and if the datapoint is within the quantile range
    col_include_in_plot = []
    for idx, row in pred_df_annot.iterrows():
        if row["SCALED_F"] >= quantile_df.loc[row["MERGED_CN_CLASS"]]["LOWER"] and row["SCALED_F"] <= quantile_df.loc[row["MERGED_CN_CLASS"]]["UPPER"]:
            col_include_in_plot.append("YES")
        else:
            col_include_in_plot.append("NO")
    pred_df_annot["PLOT"] = col_include_in_plot
    
    ### whole chromosome landscape
    ax_vio_a = sns.violinplot(
        data=pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["CORRECT_CN"] =="YES")], 
        x="MERGED_CN_CLASS", 
        y="SCALED_F", 
        ax=axis, 
        order=["WHOLE_LOSS", "NORMAL", "WHOLE_GAIN", "GAIN_4"], 
        inner=None,
        bw_adjust=.9,
        palette=["gainsboro"]#palette=["lightblue", "white", "coral", "red"]
    )

    ### manual boxplots for the whole chromosome CNVs
    box = axis.boxplot(
        [
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["MERGED_CN_CLASS"] == "WHOLE_LOSS") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"],
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["MERGED_CN_CLASS"] == "NORMAL") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"],
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["MERGED_CN_CLASS"] == "WHOLE_GAIN") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"],
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["MERGED_CN_CLASS"] == "GAIN_4") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"]
        ],
        positions=[0, 1, 2, 3],
        patch_artist=True, 
        showfliers=False,
        whiskerprops=dict(visible=False),
        widths=0.08,
        medianprops=dict(color="black", linewidth=1),  # make median line black
        capprops=dict(linewidth=0)   # hides the crossbars (caps)
    )
    # Apply colors
    for patch in box['boxes']:
        patch.set_facecolor("silver")

    ### manual boxplots for the arm level CNVs
    box = axis.boxplot(
        [
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["CN_CLASS"] == "P_ARM_LOSS") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"],
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["CN_CLASS"] == "Q_ARM_LOSS") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"],
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["CN_CLASS"] == "P_ARM_GAIN") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"],
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["CN_CLASS"] == "Q_ARM_GAIN") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"],
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["CN_CLASS"] == "ISO_P") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"],
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["CN_CLASS"] == "ISO_Q") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"]
        ],
        positions=[4 - 0.08 / 2, 4 + 0.08 / 2, 5 - 0.08 / 2, 5 + 0.08 / 2, 6 - 0.08 / 2, 6 + 0.08 / 2],
        widths=0.08,
        patch_artist=True, 
        showfliers=False,
        whiskerprops=dict(visible=False),
        capprops=dict(linewidth=0),
        medianprops=dict(color="black", linewidth=1)
    )
    # Define alternating colors
    colors = [(0, 0, 147/255), (204/255, 116/255, 0)] * 6

    # Apply colors
    for patch, color in zip(box['boxes'], colors):
        patch.set_facecolor(color)

    ax_vio_b = sns.violinplot(
        data=pred_df_annot[(pred_df_annot["MERGED_CN_CLASS"].isin(["ARM_LOSS", "ARM_GAIN", "ISO"])) & (pred_df_annot["PLOT"] == "YES") & (pred_df_annot["CORRECT_CN"] == "YES")], 
        x="MERGED_CN_CLASS", 
        y="SCALED_F", 
        ax=axis, 
        order=["WHOLE_LOSS", "NORMAL", "WHOLE_GAIN", "GAIN_4", "ARM_LOSS", "ARM_GAIN", "ISO"], 
        hue_order=["P", "Q"],
        hue="ARM", 
        split=True, 
        inner=None,
        bw_adjust=.9,
        palette=[(0, 0, 200/255), (1, 150/255, 0)]
    )


    for i in [0, 1, 2, 3, 4, 5]:
        axis.plot([-0.5, -0.5 + len(merged_cn_class_order_red) - 1], [i, i], c="gray", linestyle="dashed", zorder=0)
    axis.set_ylim(-.5, 5.5)
    axis.set_xlim(-.5, 6.5)
    xticks=[0, 1, 2, 3, 4, 5, 6]

    xtick_labels = []
    for mct, mct_label in zip(["WHOLE_LOSS", "NORMAL", "WHOLE_GAIN", "GAIN_4", "ARM_LOSS", "ARM_GAIN", "ISO"], ["Loss", "Normal", "Gain\nCN=3", "Gain\nCN=4", "Arm-loss", "Arm-gain", "Isochrom"]):
        n = pred_df_annot[(pred_df_annot["MERGED_CN_CLASS"] == mct) & (pred_df_annot["CORRECT_CN"] =="YES")].shape[0]
        xtick_labels.append(f"{mct_label}\n(n={n})")
    axis.set_xticks(xticks, xtick_labels)
    axis.set_xlabel("Chromosome CN pattern")
    axis.set_ylabel("Number of centromere in the tumor cell")
    
fig, ax = plt.subplots(figsize=(12.5, 5.5))
plot_landscape (ax)
#plt.savefig("/g/korbel/hain/centromere/paper_figures/2026_03_26_cancer_figures_landscape.svg")
plt.show()

# Analysis of arm level events

## Fitting to GMM

### Datasets

In [ ]:
### single chromosomes only
datasets = {}

for chrom in pred_df_annot["CHROM"].unique():
    for arm in ["P", "Q"]:
        cn = "1_2"
        if arm == "Q":
            cn = "2_1"
        d = np.array(pred_df_annot[
            (pred_df_annot["MERGED_CN_CLASS"] == "ARM_LOSS") & 
            (pred_df_annot["SCALED_F"].between(0, 4)) & 
            (pred_df_annot["CHROM"] == chrom) &
            (pred_df_annot["CN"] == cn) &
            (pred_df_annot["ARM"].isin([arm]))
        ].drop_duplicates(subset="SAMPLE_TUMOR")["SCALED_F"].values)
        
        if d.shape[0] >= 5:
            datasets[f"{chrom}_{arm}"] = d
        else:
            print(f"Skipping dataset {chrom}_{arm} with only {d.shape[0]} datapoints")

### GMM fit

In [ ]:
"""
Fit a fixed Gaussian Mixture Model (GMM) using non-negative least squares (NNLS).
The function enforces the sum-to-1 constraint on the mixture weights through a heavily weighted penalty row.
"""

def fit_fixed_gmm_nnls (x, loc, scale, penalty=1e3):
    
    ### prepare data
    # sorted datapoint
    data_x = np.sort(x)
    # empirical CDF - each datapoint is 1/len(data_x) up
    data_y = np.arange(1, len(data_x) + 1) / len(data_x)
    
    ### Build design matrix - each column corresponds to the CDF of a Gaussian component at the sorted datapoints
    A = scst.norm.cdf(data_x[:, None], loc=np.asarray(loc), scale=np.asarray(scale))
    
    ### Add sum-to-1 constraint through a heavily weighted row (penalty)
    A_constrained = np.vstack([A, penalty * np.ones(len(loc))])
    
    ### Prepare the constrained target vector for the nnls solver (add the penalty row)
    y_constrained = np.append(data_y, penalty)
    
    ### get weights - solve the optimization problem
    weights, _ = nnls(A_constrained, y_constrained)
    ### normalize weights to sum to 1 - because the nnls solver does not enforce the sum-to-1 constraint exactly only approximatly with the penalty row
    weights = weights / weights.sum()

    return weights

In [ ]:
### set parameters for Gaussian mixture model
n_params = 11
loc=[np.round(i, 1) for i in np.arange(0.6, 2.8, 0.2)]
scale=[0.15] * n_params

columns = [str(x) for x in loc]

In [ ]:
### store nnls fit results (kept separate from df_gmm)
data_nnls = []
index_nnls = []

for dataset in tqdm(datasets):
    w = fit_fixed_gmm_nnls(datasets[dataset], loc, scale, penalty=1e4)
    data_nnls.append(w)
    index_nnls.append(dataset)


df_gmm = pd.DataFrame(data=data_nnls, index=index_nnls, columns=columns)
df_gmm["N"] = [len(datasets[idx]) for idx in df_gmm.index]

In [ ]:
# plotting order
plot_idx = [
    'chr1_P',
    'chr3_P',
    'chr4_P',
    'chr4_Q',
    'chr5_Q',
    'chr6_Q',
    'chr8_P',
    'chr9_P',
    'chr10_P',
    'chr10_Q',
    'chr12_P',
    'chr12_Q',
    'chr16_Q',
    'chr17_P',
    'chr18_P',
    'chr18_Q',
    'chr19_P',
    'chr19_Q',
    'chr20_P']

plot_idx = [x for x in plot_idx if x in df_gmm.index]

In [ ]:
annot_data = []
for i in plot_idx:
    if i.endswith("_Q"):
        arm_color = "gold"
    else:
        arm_color = "cyan"

    n_color = [np.sqrt(df_gmm.loc[i]["N"].max()) / np.sqrt(df_gmm.loc[plot_idx]["N"].max()), 0, 0]
    n_color = [(df_gmm.loc[i]["N"].max()) / (df_gmm.loc[plot_idx]["N"].max()), 0, 0]
    
    annot_data.append({
        "IDX":i,
        "ARM":arm_color,
        #"N":n_color
    })
annot_df = pd.DataFrame(annot_data).set_index("IDX")

In [ ]:
heatmap_plot_idx = []
for idx in df_gmm.index:
    if idx.startswith("chr") and (idx.endswith("_P") or idx.endswith("_Q")) and df_gmm.at[idx, "N"] >= 10: # type: ignore
        heatmap_plot_idx.append(idx)

In [ ]:
annot_data = []
for idx in heatmap_plot_idx:
    if idx.endswith("_P"):
        annot_data.append({"IDX":idx, "ARM":(0, 0, 200/255)})
    else:
        annot_data.append({"IDX":idx, "ARM":(1, 150/255, 0)})
annot_df = pd.DataFrame(annot_data).set_index("IDX")

In [ ]:
df_gmm

In [ ]:
df_gmm["STRONG_TRUNC"] = df_gmm[["0.6", "0.8", "1.0", "1.2", "1.4"]].sum(axis=1)
df_gmm["WEAK_TRUNC"] = df_gmm[["1.6", "1.8"]].sum(axis=1)
df_gmm["NORMAL"] = df_gmm[["2.0", "2.2"]].sum(axis=1)
df_gmm["LONGER"] = df_gmm[["2.4", "2.6"]].sum(axis=1)

g = sns.clustermap(
    df_gmm[["STRONG_TRUNC", "WEAK_TRUNC", "NORMAL", "LONGER"]].loc[heatmap_plot_idx],
    col_cluster=False,
    row_cluster=False,
    cmap="viridis",
    row_colors=annot_df
)
#g.savefig("/g/korbel/hain/centromere/paper_figures/2026_03_26_cancer_figures_heatmap.svg")

### Load CDR position

In [ ]:
cdr_gao_df = pd.read_csv("Gao_CDR_position.tsv", sep="\t")

In [ ]:
data_structuring = []

for idx in df_gmm.index:
    if idx.startswith("chr"):
        chrom = idx.split("_")[0]
        arm = idx.split("_")[1]
        complete = idx
        tp53_state = "NA"
        if complete.endswith("_MUT"):
            tp53_state = "MUT"
        if complete.endswith("_WT"):
            tp53_state = "WT"
        data_structuring.append({
            "CHROM":chrom, 
            "ARM":arm, 
            "COMPLETE":complete, 
            "TP53":tp53_state, 
            "N":df_gmm.at[idx, "N"]})
data_structuring_df = pd.DataFrame(data_structuring)

In [ ]:
### dip position
col_dip_position = []

for idx, row in data_structuring_df.iterrows():
    col_dip_position.append(cdr_gao_df[cdr_gao_df["CHROM"] == row["CHROM"]]["CDR_MEDIAN"].values[0])

data_structuring_df["DIP_LOC"] = col_dip_position

In [ ]:
col_query = []
for _, row in data_structuring_df.iterrows():
    col_query.append(df_gmm.loc[row["COMPLETE"], "STRONG_TRUNC"])
data_structuring_df["STRONG_TRUNC"] = col_query

#### PLOT - Correlation

In [ ]:
def plot_correlation (axis):
    
    cdr_mean = "ASM_MEDIAN"
    
    value = "STRONG_TRUNC"
    
    for arm in ["P", "Q"]:
        
        fit_x = []
        fit_y = []
        fit_weights = []
        
        for idx, row in data_structuring_df[data_structuring_df["N"] >= 10].iterrows():
            
            if not "chr" in row["COMPLETE"]:
                continue
            if not f"_{arm}" in row["COMPLETE"]:
                continue
            if "_MUT" in row["COMPLETE"]:
                continue
            if "_WT" in row["COMPLETE"]:
                continue
            
            chrom = row["CHROM"]
            size = row["N"]
            if arm == "P":
                color = (0, 0, 200/255)
            if arm == "Q":
                color = (1, 150/255, 0)

            axis.scatter(
                row["STRONG_TRUNC"],
                row["DIP_LOC"],
                color=color,
                #s=size
            )
            ### label chromosome
            
            axis.text(
                row["STRONG_TRUNC"] + 0.01,
                row["DIP_LOC"],
                chrom,
                color=color,
                va="center"
            )
            
            if size > 0:
                fit_x.append(row["STRONG_TRUNC"])
                fit_y.append(row["DIP_LOC"])
                fit_weights.append(size)
        
        if len(fit_x) < 3:
            continue
        fit_x_C = sm.add_constant(fit_x)  # add intercept
        #model = sm.WLS(fit_y, fit_x_C, weights=fit_weights)
        model = sm.WLS(fit_y, fit_x_C)
        results = model.fit()
        
        pred = results.get_prediction(fit_x_C)
        pred_summary = pred.summary_frame(alpha=0.05)        
        sm_df = pd.DataFrame(data={"x":fit_x_C[:,1], "y":pred_summary["mean"], "lower":pred_summary["mean_ci_lower"], "upper":pred_summary["mean_ci_upper"]}).sort_values(by="x")
        
        
        
        #axis.plot(sm_df["x"], sm_df["y"], label=f"{arm.lower()}-arm loss\n$R^2$={np.round(results.rsquared, 3)}, p={np.round(results.pvalues[1], 3)}", color=color)
        axis.plot(sm_df["x"], sm_df["y"], label=f"{arm.lower()} arm loss\n$R^2$={np.round(results.rsquared, 3)}\np={np.round(results.pvalues[1], 3)}", color=color)
        #axis.fill_between(sm_df["x"], sm_df["lower"], sm_df["upper"], alpha=0.1, color=color)
    
    axis.set_ylabel("Position of the methylation dip\np-distal     <--->     q-distal")
    axis.set_xlabel("Fraction of arm losses with highly truncated centromere")
    axis.legend(loc='upper center', ncol=2)
    axis.set_ylim(0, 1)
    #axis.set_xlim(0, 0.61)
    

fig, ax = plt.subplots(figsize=(8.5, 5.5))
plot_correlation (ax)
ax.set_xlim((np.float64(-0.02692806034602134), np.float64(0.5654892672664481)))
ax.set_title("Correlation CDR vs. highly truncated centromeres")
plt.gcf().set_size_inches(7, 4)
plt.tight_layout()
plt.savefig("../plotting/plots/cancer_figures_correlation.ppcg.png")
plt.show()